# Classification with LightGBM V3.1 (oracle) — Response, Gradient-Boosted Trees, k=5

**Version:** V3.1 (k-fold validation — LightGBM ceiling check). **Role:** reference oracle for the
V3.1 scratch twin. Same `LGBMClassifier` hypers as `classification_lightgbm_V3_0.ipynb`
(`scale_pos_weight`, 500-round cap, 50-round early stopping on a 10%-of-train slice), copied verbatim
and wrapped in a 5-fold `StratifiedKFold` loop over the full 2,236-row cleaned set instead of one fixed
80/20 split. Weights → `./weights_V3_1/`.

In [1]:
import json
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

SEED = 42
np.random.seed(SEED)
os.makedirs("weights_V3_1", exist_ok=True)
print("lightgbm:", lgb.__version__)

df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = [
    "AcceptedCmp5", "AcceptedCmp3", "MntWines", "MntMeatProducts",
    "NumCatalogPurchases", "Recency", "Customer_Tenure_Days", "MntGoldProds",
    "Income", "MntFruits",
    "AcceptedCmp1", "AcceptedCmp2", "NumWebPurchases", "MntSweetProducts",
    "MntFishProducts", "NumStorePurchases", "Age",
]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
inc_idx = FEATURE_COLS.index("Income")
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

lightgbm: 4.7.0
cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


## K-fold training (k=5) — fresh LGBMClassifier per fold, own 10%-of-train early-stopping slice

In [2]:
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
X_all = X_full.values
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_full), start=1):
    Xtr_f, Xva_f = X_all[tr_idx].copy(), X_all[va_idx].copy()
    ytr_f, yva_f = y_full[tr_idx], y_full[va_idx]
    med_f = np.nanmedian(Xtr_f[:, inc_idx])
    for arr in (Xtr_f, Xva_f):
        arr[:, inc_idx] = np.where(np.isnan(arr[:, inc_idx]), med_f, arr[:, inc_idx])

    pos_f, neg_f = (ytr_f == 1).sum(), (ytr_f == 0).sum()
    scale_pos_weight_f = neg_f / pos_f

    X_tr, X_val, y_tr, y_val = train_test_split(
        Xtr_f, ytr_f, test_size=0.10, random_state=SEED, stratify=ytr_f)

    clf = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=-1,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight_f, random_state=SEED, verbosity=-1,
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], eval_names=["train", "valid"],
            eval_metric="binary_logloss", callbacks=[lgb.early_stopping(50, verbose=False)])

    prob_v = clf.predict_proba(Xva_f)[:, 1]
    pred_v = (prob_v >= 0.5).astype(int)
    fold_rows.append({
        "fold": fold,
        "best_iteration": int(clf.best_iteration_),
        "test_loss_BCE": round(float(log_loss(yva_f, prob_v)), 4),
        "accuracy": round(float(accuracy_score(yva_f, pred_v)), 4),
        "precision": round(float(precision_score(yva_f, pred_v, zero_division=0)), 4),
        "recall": round(float(recall_score(yva_f, pred_v, zero_division=0)), 4),
        "f1": round(float(f1_score(yva_f, pred_v, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(yva_f, prob_v)), 4),
    })
    print(f"fold {fold}: F1={fold_rows[-1]['f1']:.4f} BCE={fold_rows[-1]['test_loss_BCE']:.4f} best_iter={fold_rows[-1]['best_iteration']}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_loss_BCE", "accuracy", "precision", "recall", "f1", "roc_auc")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V3_1/classification_lightgbm_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 1: F1=0.6015 BCE=0.2724 best_iter=107


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 2: F1=0.5481 BCE=0.3471 best_iter=130
fold 3: F1=0.6667 BCE=0.2697 best_iter=111


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 4: F1=0.5753 BCE=0.3007 best_iter=88


/home/waroon/SecondYear/FirstSemester/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 5: F1=0.5802 BCE=0.2770 best_iter=96
 fold  best_iteration  test_loss_BCE  accuracy  precision  recall     f1  roc_auc
    1             107         0.2724    0.8817     0.6061  0.5970 0.6015   0.9068
    2             130         0.3471    0.8635     0.5441  0.5522 0.5481   0.8621
    3             111         0.2697    0.8904     0.6125  0.7313 0.6667   0.9125
    4              88         0.3007    0.8613     0.5316  0.6269 0.5753   0.8907
    5              96         0.2770    0.8770     0.5846  0.5758 0.5802   0.8955
{
  "test_loss_BCE": {
    "mean": 0.2934,
    "std": 0.0324
  },
  "accuracy": {
    "mean": 0.8748,
    "std": 0.0123
  },
  "precision": {
    "mean": 0.5758,
    "std": 0.0364
  },
  "recall": {
    "mean": 0.6166,
    "std": 0.0697
  },
  "f1": {
    "mean": 0.5944,
    "std": 0.0447
  },
  "roc_auc": {
    "mean": 0.8935,
    "std": 0.0196
  }
}
